<a href="https://colab.research.google.com/github/srihanreddy/Reinforcement-Learning/blob/main/Lab_Assignment_11_Policy_Iterative_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

# --- 1. Define the MDP ---
states = ['S0', 'S1', 'G']
non_terminal_states = ['S0', 'S1']
actions = ['Up', 'Down', 'Left', 'Right']

gamma = 0.9
r_normal = -1
r_wall = -1
r_goal = 100 # Using the parameter value from the bottom of the prompt (+100 for goal)

# Transition and reward function: transition(state, action) -> (next_state, reward)
# Deterministic Grid:
# [ S0 ] [ S1 ]
# [Wall] [  G ]
def step(state, action):
    if state == 'G':
        return 'G', 0

    if state == 'S0':
        if action == 'Up':
            return 'S0', r_wall  # Hits boundary
        elif action == 'Down':
            return 'S0', r_wall  # Hits wall (below S0 is a wall/obstacle)
        elif action == 'Left':
            return 'S0', r_wall  # Hits boundary
        elif action == 'Right':
            return 'S1', r_normal

    elif state == 'S1':
        if action == 'Up':
            return 'S1', r_wall  # Hits boundary
        elif action == 'Down':
            return 'G', r_goal    # Reaches Goal
        elif action == 'Left':
            return 'S0', r_normal
        elif action == 'Right':
            return 'S1', r_wall  # Hits boundary

    return state, 0

# --- 2. Initialize Policy and Values ---
policy = {'S0': 'Up', 'S1': 'Up'}
V = {'S0': 0.0, 'S1': 0.0, 'G': 0.0}
theta = 1e-6

# Print environment information
print("=== MDP Setup ===")
print(f"States: {states}")
print(f"Actions: {actions}")
print(f"Discount factor (gamma): {gamma}")
print(f"Normal Reward: {r_normal}, Wall Reward: {r_wall}, Goal Reward: {r_goal}\n")

=== MDP Setup ===
States: ['S0', 'S1', 'G']
Actions: ['Up', 'Down', 'Left', 'Right']
Discount factor (gamma): 0.9
Normal Reward: -1, Wall Reward: -1, Goal Reward: 100



In [4]:
# --- 3 & 4 & 5 & 6. Policy Iteration Loop ---
iteration = 0
policy_stable = False

print("=== Starting Policy Iteration ===")

while not policy_stable:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")

    # Policy Evaluation
    eval_steps = 0
    while True:
        delta = 0
        for s in non_terminal_states:
            v_old = V[s]
            a = policy[s]
            s_next, r = step(s, a)
            V[s] = r + gamma * V[s_next]
            delta = max(delta, abs(v_old - V[s]))
        eval_steps += 1
        if delta < theta:
            break

    print(f"Policy Evaluation converged in {eval_steps} steps.")
    print(f"Current State Values V: S0: {V['S0']:.2f}, S1: {V['S1']:.2f}")

    # Policy Improvement
    policy_stable = True
    for s in non_terminal_states:
        old_action = policy[s]

        # Calculate Q(s, a) for all actions
        best_action = None
        best_q = -float('inf')

        for a in actions:
            s_next, r = step(s, a)
            q_val = r + gamma * V[s_next]
            if q_val > best_q:
                best_q = q_val
                best_action = a

        policy[s] = best_action
        if old_action != best_action:
            policy_stable = False

    print(f"Updated Policy: S0 -> {policy['S0']}, S1 -> {policy['S1']}")

print("\nOptimal Policy Found!")

=== Starting Policy Iteration ===

--- Iteration 1 ---
Policy Evaluation converged in 133 steps.
Current State Values V: S0: -10.00, S1: -10.00
Updated Policy: S0 -> Up, S1 -> Down

--- Iteration 2 ---
Policy Evaluation converged in 2 steps.
Current State Values V: S0: -10.00, S1: 100.00
Updated Policy: S0 -> Right, S1 -> Down

--- Iteration 3 ---
Policy Evaluation converged in 2 steps.
Current State Values V: S0: 89.00, S1: 100.00
Updated Policy: S0 -> Right, S1 -> Down

Optimal Policy Found!


In [5]:
# --- 7. Final Results and Bellman Optimality Verification ---
print("\n=== Final Results ===")
data_final = []
for s in states:
    opt_act = policy[s] if s != 'G' else 'Terminal'
    data_final.append([s, f"{V[s]:.2f}", opt_act])

df_final = pd.DataFrame(data_final, columns=['State', 'Optimal Value V*(s)', 'Optimal Action'])
display(df_final)

print("\n=== Action-Value Table Q*(s, a) ===")
q_data = {}
for s in non_terminal_states:
    q_data[s] = {}
    for a in actions:
        s_next, r = step(s, a)
        q_data[s][a] = r + gamma * V[s_next]

df_q = pd.DataFrame(q_data).T
display(df_q)

print("\n=== Bellman Optimality Verification ===")
for s in non_terminal_states:
    max_q = max(df_q.loc[s])
    v_star = V[s]
    print(f"State {s}: V*(s) = {v_star:.2f} | max_a Q*(s, a) = {max_q:.2f} | Equal: {np.isclose(v_star, max_q)}")


=== Final Results ===


,State,Optimal Value V*(s),Optimal Action
0,S0,89.00,Right
1,S1,100.00,Down
2,G,0.00,Terminal



=== Action-Value Table Q*(s, a) ===


,Up,Down,Left,Right
S0,79.1,79.1,79.1,89.0
S1,89.0,100.0,79.1,89.0



=== Bellman Optimality Verification ===
State S0: V*(s) = 89.00 | max_a Q*(s, a) = 89.00 | Equal: True
State S1: V*(s) = 100.00 | max_a Q*(s, a) = 100.00 | Equal: True
